<a href="https://colab.research.google.com/github/not7devansh/Cataract-Detector-From-Webcam/blob/main/Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
device='cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [ ]:
import kagglehub
nandanp6_cataract_image_dataset_path = kagglehub.dataset_download('nandanp6/cataract-image-dataset')

print('Data source import complete.')

Using Colab cache for faster access to the 'cataract-image-dataset' dataset.
Data source import complete.


In [ ]:
import os

print("Dataset path:", nandanp6_cataract_image_dataset_path)


for root, dirs, files in os.walk(nandanp6_cataract_image_dataset_path):
    print(f"Folder: {root} | Files count: {len(files)}")

Dataset path: /kaggle/input/cataract-image-dataset
Folder: /kaggle/input/cataract-image-dataset | Files count: 0
Folder: /kaggle/input/cataract-image-dataset/processed_images | Files count: 0
Folder: /kaggle/input/cataract-image-dataset/processed_images/test | Files count: 0
Folder: /kaggle/input/cataract-image-dataset/processed_images/test/normal | Files count: 60
Folder: /kaggle/input/cataract-image-dataset/processed_images/test/cataract | Files count: 61
Folder: /kaggle/input/cataract-image-dataset/processed_images/train | Files count: 0
Folder: /kaggle/input/cataract-image-dataset/processed_images/train/normal | Files count: 246
Folder: /kaggle/input/cataract-image-dataset/processed_images/train/cataract | Files count: 245


In [ ]:
class CatModel(nn.Module):
  def __init__(self,input=3,hidden_unit=16,output=2):
    super().__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=input,out_channels=hidden_unit,kernel_size=(3,3),stride=1,padding=1,bias=False),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_unit,out_channels=hidden_unit,kernel_size=(3,3),stride=1,padding=1,bias=False),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.conv_block_2=nn.Sequential(
        nn.Conv2d(in_channels=hidden_unit,out_channels=hidden_unit,kernel_size=(3,3),stride=1,padding=1,bias=False),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_unit,out_channels=hidden_unit,kernel_size=(3,3),stride=1,padding=1,bias=False),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=16*32*32,out_features=2),
    )
  def forward(self,x):
    x=self.conv_block_1(x)
    x=self.conv_block_2(x)
    return self.classifier(x)
model=CatModel(input=3,hidden_unit=16,output=2)
model=model.to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)


In [ ]:

!pip install -q mediapipe

import os
import cv2 as cv
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1
)
detector = vision.FaceLandmarker.create_from_options(options)


cleaned_base = "./cleaned_dataset"
for split in ["train", "test"]:
    for label in ["cataract", "normal"]:
        os.makedirs(os.path.join(cleaned_base, split, label), exist_ok=True)

def crop_and_resize(src_folder, dest_folder):
    for fname in os.listdir(src_folder):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        img_path = os.path.join(src_folder, fname)
        img = cv.imread(img_path)
        if img is None:
            continue

        h, w, _ = img.shape
        rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

        result = detector.detect(mp_img)

        if result.face_landmarks:
            lm = result.face_landmarks[0]
            lx_min = min(int(lm[469].x * w), int(lm[471].x * w))
            lx_max = max(int(lm[469].x * w), int(lm[471].x * w))
            ly_min = min(int(lm[470].y * h), int(lm[472].y * h))
            ly_max = max(int(lm[470].y * h), int(lm[472].y * h))

            pad_x = int((lx_max - lx_min) * 0.6)
            pad_y = int((ly_max - ly_min) * 0.6)

            x1, y1 = max(0, lx_min - pad_x), max(0, ly_min - pad_y)
            x2, y2 = min(w, lx_max + pad_x), min(h, ly_max + pad_y)

            crop = img[y1:y2, x1:x2]
        else:

            crop = img

        if crop.size > 0:
            resized = cv.resize(crop, (128, 128))
            cv.imwrite(os.path.join(dest_folder, fname), resized)

for split in ["train", "test"]:
    for label in ["cataract", "normal"]:
        src = os.path.join(nandanp6_cataract_image_dataset_path, "processed_images", split, label)
        dest = os.path.join(cleaned_base, split, label)
        crop_and_resize(src, dest)

print("Saari images 128x128 eye crops mein convert ho chuki hain!")

Saari images 128x128 eye crops mein convert ho chuki hain!


In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Paths
train_dir = "./cleaned_dataset/train"
test_dir = "./cleaned_dataset/test"

# 3. Transforms (Training gets augmentation; Testing gets deterministic normalization)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 4. ImageFolder Datasets & DataLoaders
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Class Mapping: {train_dataset.class_to_idx}")  # {'cataract': 0, 'normal': 1}
print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")

Using device: cuda
Class Mapping: {'cataract': 0, 'normal': 1}
Train samples: 491 | Test samples: 121


In [ ]:
epochs = 20
for epoch in range(epochs):
    train_loss, train_acc = 0, 0
    model.train()
    for image, label in train_loader:
        image, label = image.to(device), label.to(device)

        optimizer.zero_grad()

        y_pred = model(image)
        loss = loss_fn(y_pred, label)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Accuracy calculation
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == label).sum().item() / len(y_pred)

    train_loss = train_loss / len(train_loader)
    train_acc = train_acc / len(train_loader)
    print(f"Epoch: {epoch+1} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")

    model.eval()
    test_loss, test_acc = 0, 0

    with torch.no_grad():
        for image, label in test_loader:
            image, label = image.to(device), label.to(device)

            y_test = model(image)
            loss = loss_fn(y_test, label)
            test_loss += loss.item()

            y_pred_class = torch.argmax(torch.softmax(y_test, dim=1), dim=1)
            test_acc += (y_pred_class == label).sum().item() / len(y_pred_class)

    test_loss = test_loss / len(test_loader)
    test_acc = test_acc / len(test_loader)
    print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc*100:.2f}%\n")

Epoch: 1 | Train Loss: 0.6454 | Train Acc: 64.55%
Test Loss: 0.5758 | Test Acc: 79.60%

Epoch: 2 | Train Loss: 0.5484 | Train Acc: 72.29%
Test Loss: 0.4407 | Test Acc: 82.81%

Epoch: 3 | Train Loss: 0.4343 | Train Acc: 84.00%
Test Loss: 0.3238 | Test Acc: 86.11%

Epoch: 4 | Train Loss: 0.3668 | Train Acc: 85.59%
Test Loss: 0.3281 | Test Acc: 84.72%

Epoch: 5 | Train Loss: 0.3415 | Train Acc: 86.00%
Test Loss: 0.3584 | Test Acc: 85.94%

Epoch: 6 | Train Loss: 0.3176 | Train Acc: 86.60%
Test Loss: 0.2887 | Test Acc: 91.58%

Epoch: 7 | Train Loss: 0.3014 | Train Acc: 89.11%
Test Loss: 0.2711 | Test Acc: 89.84%

Epoch: 8 | Train Loss: 0.2424 | Train Acc: 89.83%
Test Loss: 0.2914 | Test Acc: 91.58%

Epoch: 9 | Train Loss: 0.2223 | Train Acc: 91.94%
Test Loss: 0.2897 | Test Acc: 87.07%

Epoch: 10 | Train Loss: 0.2334 | Train Acc: 90.03%
Test Loss: 0.3286 | Test Acc: 90.62%

Epoch: 11 | Train Loss: 0.2180 | Train Acc: 91.44%
Test Loss: 0.3000 | Test Acc: 90.19%

Epoch: 12 | Train Loss: 0.1891

In [ ]:
import torch
from google.colab import files

torch.save(model.state_dict(), "catmodel_weights.pth")
print("Model saved!")
files.download("catmodel_weights.pth")

Model saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>